# Lamoda · Проверка верхней части рейтинга

Запуск из корня репозитория с архивом соревнования или переменной `LAMODA_DATA`. В окружение предыдущей модели добавьте CatBoost 1.2.10 и XGBoost CPU 3.4.1. Ноутбук формирует апрельский CSV и ZIP. Кандидата выбираем по февралю, проверяем в марте. Если он уступает контролю хотя бы раз, итоговый файл сохраняет прогноз квартальной модели.

In [ ]:
%run ./lamoda_context.ipynb

### Подготовка матрицы и проверка прогнозов

Сохранённые пять январских фолдов содержат прогнозы для клиентов, исключённых из обучения всех базовых моделей, включая отбор признаков и ранжировщик. Февраль и март используют обучение на предыдущих месяцах. Для полного пересчёта прогнозов задайте `rebuild = True` и используйте новый каталог кэша. При изменении данных, признаков или гиперпараметров также выберите новый каталог.

In [ ]:
from pathlib import Path
from research.tail_inputs import prepare
from research.run_tail import run
from research.tail_review import review
import shutil, json

experiment_cache = Path('.checkpoints/tail6_notebook')
rebuild = False
prepare(globals(),experiment_cache,rebuild_scores=rebuild)

In [ ]:
if not rebuild:
    for fold in range(5):
        filename = f'january_fold_{fold}.parquet'
        shutil.copyfile(Path('research/oof') / filename,experiment_cache / filename)
prediction, selected_model = run(experiment_cache)
february_choice = json.loads(Path('research/results/tail_february_choice.json').read_text())['model']
review(experiment_cache,february_choice)
print('Модель для апреля',selected_model)
display(pd.read_csv('research/results/tail_models.csv').query('model == @february_choice'))

### Проверенный прогноз в порядке образца Kaggle

In [ ]:
submission_tail = write_submission(prediction,'submission_tail')
assert len(submission_tail) == len(test) == 25000
print('Строк',len(submission_tail),'· CSV и ZIP проверены')